# Verify global_mean_pool Fix

Quick test to verify the pooling optimization is working.

In [9]:
import sys
import time
import importlib
from pathlib import Path

# Path setup
BASE_PATH = Path('../..')
sys.path.insert(0, str(BASE_PATH))

# Force reimport of models module
if 'models' in sys.modules:
    del sys.modules['models']

import torch
import numpy as np
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool

# Now import models (fresh)
import models
from models import SurfaceCodeSampler, SparseGraph, GraphSAGE, GraphSAGEModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Models module loaded from: {models.__file__}")

Device: cuda
Models module loaded from: g:\My Drive\Research\QEC\quantum-error-correction\quantum-error-correction\code\gSAGE\speedup\..\..\models.py


## 1. Check if global_mean_pool is in the models module

In [10]:
# Check if global_mean_pool is imported in models.py
import inspect

# Get the source of the forward method
source = inspect.getsource(GraphSAGEModel.forward)
print("GraphSAGEModel.forward() source code:")
print("="*60)
print(source)
print("="*60)

if 'global_mean_pool' in source:
    print("\n✓ global_mean_pool IS being used!")
elif 'for i in range(batch_size)' in source:
    print("\n✗ OLD for-loop is still being used!")
else:
    print("\n? Unknown pooling method")

GraphSAGEModel.forward() source code:
    def forward(self, data) -> torch.Tensor:
        x, edge_index = data.x, data.edge_index
        # Convert edge_attr [N,1] to edge_weight [N] for weighted aggregation
        edge_weight = data.edge_attr.view(-1) if hasattr(data, 'edge_attr') and data.edge_attr is not None else None
        batch = data.batch if hasattr(data, 'batch') and data.batch is not None else torch.zeros(x.size(0), dtype=torch.long, device=x.device)

        # Apply WeightedSAGEConv layers with batch normalization, activation, and dropout
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index, edge_weight)
            x = bn(x)
            x = F.silu(x)
            x = self.dropout(x)

        # Global mean pooling: aggregate node features to graph-level (optimized)
        x_pooled = global_mean_pool(x, batch)

        # Classification layers
        x = self.fc1(x_pooled)
        x = F.silu(x)
        x = self.dropout(x)
        x = self.

## 2. Direct Speed Comparison

In [11]:
# Generate test data
print("Generating test data...")
graph_builder = SparseGraph(k_neighbors=6, device=device)
sampler = SurfaceCodeSampler(p=0.005, device=device)

# Test with d=7
d = 7
BATCH_SIZE = 64
NUM_SAMPLES = 1000

detections_batch, labels_batch = sampler.sample(d=d, num_samples=NUM_SAMPLES)
graphs = []
for i in range(NUM_SAMPLES):
    graph = graph_builder.to_pyg(detections_batch[i], labels_batch[i])
    graphs.append(graph)

loader = DataLoader(graphs, batch_size=BATCH_SIZE, shuffle=False)
print(f"Created {NUM_SAMPLES} graphs for d={d}")

Generating test data...
SparseGraph initialized:
  K neighbors: 6
  Device: cuda
  Mode: Dynamic (supports any code distance)
SurfaceCodeSampler initialized:
  Default error rate: 0.005
  Device: cuda
  Mode: Dynamic (supports any code distance)
Created 1000 graphs for d=7


In [12]:
# Load model
models_dir = BASE_PATH / "gSAGE" / "distances" / "models" / "revised_training"
model_path = models_dir / f"d{d}.pt"

model_wrapper = GraphSAGE(nickname=f'd{d}', device=device)
model_wrapper.load(str(model_path))
model = model_wrapper.model
model.eval()
print(f"Model loaded from {model_path}")

GraphSAGE initialized: GraphSAGE(nickname='d7', in_channels=5, hidden_dim=128, num_layers=4)
Model loaded: GraphSAGE(nickname='d7', in_channels=5, hidden_dim=128, num_layers=5, aggr='max', loaded_from='d7.pt')
Model loaded from ..\..\gSAGE\distances\models\revised_training\d7.pt


In [13]:
# Benchmark the actual model forward pass
print("\nBenchmarking model forward pass...")

# Warmup
with torch.no_grad():
    for batch in loader:
        batch = batch.to(device)
        _ = model(batch)
        break
torch.cuda.synchronize()

# Time it
times = []
with torch.no_grad():
    for batch in loader:
        batch = batch.to(device)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        _ = model(batch)
        torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)

print(f"\nModel forward pass timing (batch_size={BATCH_SIZE}):")
print(f"  Mean: {np.mean(times):.3f} ms")
print(f"  Std:  {np.std(times):.3f} ms")
print(f"  Throughput: {BATCH_SIZE * 1000 / np.mean(times):.0f} samples/sec")


Benchmarking model forward pass...

Model forward pass timing (batch_size=64):
  Mean: 3.228 ms
  Std:  0.321 ms
  Throughput: 19829 samples/sec


## 3. Compare Old vs New Pooling Directly

## 4. Test Distance 13 (largest model)

In [18]:
# Test d=13 (the largest/slowest distance)
print("Testing d=13...")

d13 = 13
NUM_SAMPLES_13 = 500  # Fewer samples since d=13 is larger

# Generate data
detections_13, labels_13 = sampler.sample(d=d13, num_samples=NUM_SAMPLES_13)
graphs_13 = []
for i in range(NUM_SAMPLES_13):
    graph = graph_builder.to_pyg(detections_13[i], labels_13[i])
    graphs_13.append(graph)

loader_13 = DataLoader(graphs_13, batch_size=BATCH_SIZE, shuffle=False)
print(f"Created {NUM_SAMPLES_13} graphs for d={d13}")

# Load d=13 model
model_path_13 = models_dir / f"d{d13}.pt"
model_wrapper_13 = GraphSAGE(nickname=f'd{d13}', device=device)
model_wrapper_13.load(str(model_path_13))
model_13 = model_wrapper_13.model
model_13.eval()

# Warmup
with torch.no_grad():
    for batch in loader_13:
        batch = batch.to(device)
        _ = model_13(batch)
        break
torch.cuda.synchronize()

# Benchmark
times_13 = []
with torch.no_grad():
    for batch in loader_13:
        batch = batch.to(device)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        _ = model_13(batch)
        torch.cuda.synchronize()
        times_13.append((time.perf_counter() - t0) * 1000)

print(f"\n=== d=13 Results ===")
print(f"Model forward pass timing (batch_size={BATCH_SIZE}):")
print(f"  Mean: {np.mean(times_13):.3f} ms")
print(f"  Std:  {np.std(times_13):.3f} ms")
print(f"  Throughput: {BATCH_SIZE * 1000 / np.mean(times_13):.0f} samples/sec")
print(f"\nPrevious throughput was ~2,288 samples/sec")
print(f"Speedup: {(BATCH_SIZE * 1000 / np.mean(times_13)) / 2288:.1f}x faster!")

Testing d=13...
Created 500 graphs for d=13
GraphSAGE initialized: GraphSAGE(nickname='d13', in_channels=5, hidden_dim=128, num_layers=4)
Model loaded: GraphSAGE(nickname='d13', in_channels=5, hidden_dim=128, num_layers=5, aggr='max', loaded_from='d13.pt')

=== d=13 Results ===
Model forward pass timing (batch_size=64):
  Mean: 3.069 ms
  Std:  0.439 ms
  Throughput: 20853 samples/sec

Previous throughput was ~2,288 samples/sec
Speedup: 9.1x faster!


In [15]:
import torch.nn.functional as F

# Get one batch
batch_data = next(iter(loader)).to(device)

# Run through conv layers to get x and batch tensors
with torch.no_grad():
    x = batch_data.x
    edge_index = batch_data.edge_index
    edge_weight = batch_data.edge_attr.view(-1) if batch_data.edge_attr is not None else None
    batch = batch_data.batch
    
    for conv, bn in zip(model.convs, model.bns):
        x = conv(x, edge_index, edge_weight)
        x = bn(x)
        x = F.silu(x)

print(f"After conv layers: x.shape = {x.shape}")
print(f"batch tensor: {batch.shape}, unique values: {batch.unique().shape[0]}")

After conv layers: x.shape = torch.Size([1494, 128])
batch tensor: torch.Size([1494]), unique values: 64


In [16]:
# Benchmark OLD method (for-loop)
def old_pooling(x, batch, batch_size):
    x_pooled = torch.zeros(batch_size, x.size(1), device=x.device)
    for i in range(batch_size):
        mask = (batch == i)
        if mask.sum() > 0:
            x_pooled[i] = x[mask].mean(dim=0)
    return x_pooled

# Benchmark NEW method (global_mean_pool)
def new_pooling(x, batch):
    return global_mean_pool(x, batch)

NUM_ITER = 100

# OLD method
torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(NUM_ITER):
    _ = old_pooling(x, batch, BATCH_SIZE)
torch.cuda.synchronize()
old_time = (time.perf_counter() - t0) * 1000 / NUM_ITER

# NEW method
torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(NUM_ITER):
    _ = new_pooling(x, batch)
torch.cuda.synchronize()
new_time = (time.perf_counter() - t0) * 1000 / NUM_ITER

print(f"Pooling comparison (d={d}, batch_size={BATCH_SIZE}):")
print(f"  OLD (for-loop):       {old_time:.3f} ms")
print(f"  NEW (global_mean_pool): {new_time:.3f} ms")
print(f"  Speedup: {old_time/new_time:.1f}x")

Pooling comparison (d=7, batch_size=64):
  OLD (for-loop):       23.037 ms
  NEW (global_mean_pool): 0.233 ms
  Speedup: 99.0x


In [ ]:
# Verify outputs are the same
old_result = old_pooling(x, batch, BATCH_SIZE)
new_result = new_pooling(x, batch)

diff = (old_result - new_result).abs().max().item()
print(f"Max difference between methods: {diff:.2e}")
print(f"Results match: {diff < 1e-5}")

Max difference between methods: 9.54e-07
Results match: True
